# Localization - Working with Different Languages

This notebook shows how to:
1. Set the localization language
2. Retrieve localized texts
3. Export all texts as JSON
4. Work with Text objects

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter
import json

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

print(f"Loaded {len(texts.elements)} text entries")

Loaded 33270 text entries


## Available Languages

Anno 117 supports multiple languages. Check which ones are available:

In [2]:
# Get first text entry to see available languages
sample_text = next(iter(texts.elements.values()))
available_languages = list(sample_text.values.keys())

print("Available languages:")
for lang in available_languages:
    print(f"  - {lang}")

Available languages:
  - english
  - german
  - spanish
  - french
  - italian
  - japanese
  - korean
  - polish
  - brazilian
  - russian
  - simplified_chinese
  - traditional_chinese


## Get Standard values

In [3]:
item = assets.get(80510) # get some asset
name = item.text # Get the localized text. This is a Text object and not a str!.

Manual specify language for conversion:

In [4]:

# Direct access to values dict
english_name = name.values.get('english', 'N/A')
german_name = name.values.get('german', 'N/A')

print("Asset text:")
print(f"  English: {english_name}")
print(f"  German: {german_name}")



Asset text:
  English: Zorsines, Sarmatian Swordshaper
  German: Zorsines, sarmatischer Schwertformer


Set the Localization Language

Use `StandardTextConverter` to set the language for automatic text conversion.

In [5]:
# Set language to German
LANGUAGE = "german"
texts.converter = StandardTextConverter(LANGUAGE)

print(f"Language set to: {LANGUAGE}")

Language set to: german


use the global set language to get localized text using the bracket notation:

In [6]:
# Using converter (respects current language setting)
converted = name()  # Uses StandardTextConverter
print(f"  Converted: {converted}")

  Converted: Zorsines, sarmatischer Schwertformer


In [7]:
# Method 2: Direct lookup by text ID
def get_text(text_id, language='english'):
    """Get text by ID in specific language."""
    text_obj = texts.elements.get(text_id)
    if text_obj:
        return text_obj.values.get(language, 'N/A')
    return None

# Example: Get localized rarity text
rarity_attr = item.Item.Rarity
if hasattr(rarity_attr, 'ui_text') and rarity_attr.ui_text:
    rarity_en = rarity_attr.ui_text.values.get('english') # manual translation
    rarity_de = rarity_attr.ui_text() # TextConverter
    print(f"\nRarity:")
    print(f"  English: {rarity_en}")
    print(f"  German: {rarity_de}")


Rarity:
  English: Legendary
  German: Legendär


## Export Text as JSON for all Languages

Export all localized texts for a specific asset:

In [8]:
def export_texts_to_json(asset):
    """Export all texts for to a dictionary."""
    
    return json.dumps(asset.text.values)

print(export_texts_to_json(item))
for lang, txt in item.text.values.items():
     print(f"{lang:10s}: {txt}")

{"english": "Zorsines, Sarmatian Swordshaper", "german": "Zorsines, sarmatischer Schwertformer", "spanish": "Zorsines, espadero s\u00e1rmata", "french": "Zorsines, fa\u00e7onneur d'\u00e9p\u00e9es sarmate", "italian": "Zorsines, armaiolo sarmato", "japanese": "\u200b\u30b5\u30eb\u30de\u30c6\u30a3\u30a2\u200b\u306e\u200b\u5263\u200b\u5320-\u200b\u30be\u30eb\u30b7\u30cd\u30b9", "korean": "\u200b\uc870\ub974\uc2dc\ub124\uc2a4, \u200b\uc0ac\ub974\ub9c8\ud2f0\uc544\uc758 \u200b\uac80 \u200b\uc7a5\uc778", "polish": "Zorsines, Sarmacki miecznik", "brazilian": "Zorsines, moldador de espadas s\u00e1rmata", "russian": "\u0417\u043e\u0440\u0441\u0438\u043d, \u0441\u0430\u0440\u043c\u0430\u0442\u0441\u043a\u0438\u0439 \u043c\u0435\u0447\u043d\u0438\u043a", "simplified_chinese": "\u200b\u201c\u8428\u200b\u9a6c\u200b\u63d0\u200b\u94f8\u200b\u5251\u200b\u5e08\u201d\u200b\u4f50\u200b\u5c14\u200b\u897f\u5185\u200b\u65af", "traditional_chinese": "\u200b\u300c\u85a9\u200b\u99ac\u200b\u63d0\u200b\u9444\u2

In [10]:
# Save to JSON file
from pathlib import Path

output_dir = Path("results/example")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "texts_english_sample.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_texts_to_json(item), f, indent=2, ensure_ascii=False)

print(f"Saved to: {output_file}")

Saved to: results\example\texts_english_sample.json


## Working with Text Formatting

Some texts have placeholders that need to be filled with values:

In [11]:
# Find a text with placeholders
# BuffAdditionalFactoryOutput uses placeholders like {amount}, {product}, {cycle}

# Get a buff with formatted text
buff_asset = assets.get(51283)  # Has AdditionalOutput
if buff_asset:
    additional_output = buff_asset.find("FactoryUpgrade.AdditionalOutput")
    if additional_output and len(additional_output) > 0:
        # Get the first output entry
        first_output = list(additional_output)[0]
        
        # The buff_ui property handles formatting automatically
        buff_ui = first_output.buff_ui
        if buff_ui:
            print("Formatted buff text:")
            print(f"  {buff_ui}")

Formatted buff text:
  Zusätzlich 1 t Flachs alle 10 Zyklen


## Next Steps

- `03_iterate_buildings.ipynb` - Work with production buildings
- `08_localize_literals.ipynb` - Localize dataset literals (rarity, item types, etc.)